In [1]:
# ==========================================================
# Cell 0 — Imports + Config (AJUSTE AQUI)
# ==========================================================
import os, json, time
from pathlib import Path
import numpy as np

# ---- Ajuste os caminhos ----
DDPM_ALL_PY     = Path("./ddpm_all.py")                         # seu código base
DDPM_CKPT_ROOT  = Path("DDPM-SAVEDMODELS")                      # pesos por classe
FILTER_ROOT     = Path("FAISS-64-54K-RESULTS")  
OUT_ROOT        = Path("DDPM-FILTERED-KNN-54K")                # saída final

# ---- Geração ----
N_FINAL_PER_CLASS = 5400      # número FINAL por classe
GEN_BATCH         = 32        # batch por iteração
TOTAL_TIMESTEPS   = 1000
IMG_SIZE          = 64
IMG_CHANNELS      = 3
CLIP_MIN          = -1.0
CLIP_MAX          = 1.0

OUT_ROOT.mkdir(parents=True, exist_ok=True)

assert DDPM_ALL_PY.exists(), f"Não achei {DDPM_ALL_PY}. Coloque o ddpm_all.py ao lado do notebook."
assert DDPM_CKPT_ROOT.exists(), f"Não achei {DDPM_CKPT_ROOT}."
assert FILTER_ROOT.exists(), f"Não achei {FILTER_ROOT}."

print("DDPM_ALL_PY:", DDPM_ALL_PY.resolve())
print("DDPM_CKPT_ROOT:", DDPM_CKPT_ROOT.resolve())
print("FILTER_ROOT:", FILTER_ROOT.resolve())
print("OUT_ROOT:", OUT_ROOT.resolve())

# ==========================================================
# Cell 1 — Importa build_model e GaussianDiffusion do ddpm_all.py
# ==========================================================
import sys

sys.path.append(str(DDPM_ALL_PY.parent.resolve()))
from ddpm_all import build_model, GaussianDiffusion

print("OK: build_model e GaussianDiffusion importados.")


DDPM_ALL_PY: /tf/2025/ddpm_all.py
DDPM_CKPT_ROOT: /tf/2025/DDPM-SAVEDMODELS
FILTER_ROOT: /tf/2025/FAISS-64-54K-RESULTS
OUT_ROOT: /tf/2025/DDPM-FILTERED-KNN-54K


2026-01-05 12:29:54.976875: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-05 12:29:55.249796: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Versão do Tensorflow: 2.16.1
GPU disponivel: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Número de GPUs: 1


2026-01-05 12:29:57.358346: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-05 12:29:57.400118: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-05 12:29:57.400137: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-05 12:29:57.405829: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-05 12:29:57.405851: I external/local_xla/xla/stream_executor

[OK] Pesos da EMA carregados de: DDPM-SAVEDMODELS/BULKCARRIER/unet_ema.weights.h5
OK: build_model e GaussianDiffusion importados.


In [2]:
# ==========================================================
# Cell 2 — TensorFlow setup + hiperparâmetros do UNet (um só lugar)
# ==========================================================
import tensorflow as tf
from tensorflow import keras

print("TF:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus if gpus else "nenhuma (CPU)")

# Hiperparâmetros do UNet (precisam bater com o que foi usado ao treinar)
batch_size = 32
total_timesteps = TOTAL_TIMESTEPS
norm_groups = 8
learning_rate = 2e-4

first_conv_channels = 64
channel_multiplier = [1, 2, 4, 8]
has_attention = [False, False, True, True]
num_res_blocks = 2

# ==========================================================
# Cell 3 — Descobrir CLASSES automaticamente (evita mismatch)
# ==========================================================
def list_subdirs(root: Path):
    return sorted([d.name for d in root.iterdir() if d.is_dir()])

classes_ckpt  = set(list_subdirs(DDPM_CKPT_ROOT))
classes_filt  = set(list_subdirs(FILTER_ROOT))
classes = sorted(classes_ckpt & classes_filt)

print("Classes encontradas (interseção ckpt ∩ filtro):")
print(classes)

# Se você quiser forçar uma lista manual, descomente:
# classes = ["BULKCARRIER", "CONTAINERSHIP", "PASSENGERSHIP", ...]
# print("Classes FORÇADAS:", classes)

assert len(classes) > 0, "Nenhuma classe em comum entre DDPM_SAVEDMODELS e FILTER_ROOT."


TF: 2.16.1
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Classes encontradas (interseção ckpt ∩ filtro):
['BULKCARRIER', 'CONTAINERSHIP', 'GENERALCARGO', 'OILPRODUCTSTANKER', 'PASSENGERSSHIP', 'TANKER', 'TRAWLER', 'TUG', 'VEHICLESCARRIER', 'YACHT']


In [3]:
# ==========================================================
# Cell 4 — Carregar EMA por classe (somente o que é necessário)
# ==========================================================
def load_ema_network_for_class(class_name: str):
    ckpt_dir = DDPM_CKPT_ROOT / class_name
    ema_weights = ckpt_dir / "unet_ema.weights.h5"
    cfg_path = ckpt_dir / "config.json"

    if cfg_path.exists():
        with open(cfg_path, "r") as f:
            saved_cfg = json.load(f)
        assert saved_cfg["img_size"] == IMG_SIZE and saved_cfg["img_channels"] == IMG_CHANNELS, \
            f"Config difere para {class_name}: {saved_cfg} vs IMG_SIZE/CHANNELS atuais."

    ema_network = build_model(
        img_size=IMG_SIZE,
        img_channels=IMG_CHANNELS,
        widths=[first_conv_channels * m for m in channel_multiplier],
        has_attention=has_attention,
        num_res_blocks=num_res_blocks,
        norm_groups=norm_groups,
        activation_fn=keras.activations.swish,
    )

    if not ema_weights.exists():
        raise FileNotFoundError(f"Não encontrei {ema_weights}")
    ema_network.load_weights(str(ema_weights))
    print(f"[OK] EMA carregada: {ema_weights}")

    gdf_util = GaussianDiffusion(
        timesteps=total_timesteps,
        clip_min=CLIP_MIN,
        clip_max=CLIP_MAX,
    )
    return ema_network, gdf_util


In [4]:
# ==========================================================
# Cell 5 — Filtro 64: TorchMetrics backbone + FAISS + threshold
# ==========================================================
import faiss
import torch
from torchmetrics.image.fid import FrechetInceptionDistance

DEVICE_TORCH = "cuda" if torch.cuda.is_available() else "cpu"
print("TORCH DEVICE:", DEVICE_TORCH)

# Backbone igual ao usado no "treino do filtro 64"
fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE_TORCH)
fid_metric.eval()

if hasattr(fid_metric, "inception"):
    inception_net = fid_metric.inception
elif hasattr(fid_metric, "feature_network"):
    inception_net = fid_metric.feature_network
else:
    raise RuntimeError("Não achei a rede inception dentro do torchmetrics FID.")

inception_net.eval().to(DEVICE_TORCH)

def load_filter(class_name: str):
    class_dir = FILTER_ROOT / class_name
    index = faiss.read_index(str(class_dir / "faiss_index_l2.index"))
    with open(class_dir / "filter_meta.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    k = int(meta["k"])
    thr_dist = float(meta["threshold_dist_mean_knn"])
    return index, k, thr_dist, meta

@torch.no_grad()
def extract_features_from_ddpm_uint8(imgs_uint8: np.ndarray) -> np.ndarray:
    """
    imgs_uint8: [B,64,64,3] uint8
    retorna: [B,2048] float32
    """
    x = torch.from_numpy(imgs_uint8).permute(0, 3, 1, 2).contiguous()  # [B,3,64,64] uint8
    x = x.to(DEVICE_TORCH)
    feats = inception_net(x)
    if isinstance(feats, (tuple, list)):
        feats = feats[0]
    return feats.detach().cpu().numpy().astype(np.float32)

def knn_mean_distance(index, query_feats: np.ndarray, k: int) -> np.ndarray:
    # FAISS L2 retorna L2^2
    D2, _ = index.search(query_feats, k)
    D = np.sqrt(np.maximum(D2, 0.0))
    return D.mean(axis=1)


TORCH DEVICE: cuda


In [5]:
# ==========================================================
# Cell 6 — DDPM sampling em batch + salvar imagem
# ==========================================================
from PIL import Image

def ddpm_sample_batch(ema_network, gdf_util, num_img: int) -> np.ndarray:
    """
    Gera um batch do DDPM e devolve uint8 [B,64,64,3]
    """
    samples = tf.random.normal(
        shape=(num_img, IMG_SIZE, IMG_SIZE, IMG_CHANNELS),
        dtype=tf.float32
    )

    for t in reversed(range(0, total_timesteps)):
        tt = tf.cast(tf.fill(num_img, t), dtype=tf.int64)
        pred_noise = ema_network.predict([samples, tt], verbose=0, batch_size=num_img)
        samples = gdf_util.p_sample(pred_noise, samples, tt, clip_denoised=True)

    imgs = tf.clip_by_value(samples * 127.5 + 127.5, 0.0, 255.0).numpy().astype(np.uint8)
    return imgs

def save_uint8_image(img_uint8: np.ndarray, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(img_uint8, mode="RGB").save(str(path))


In [6]:
# ==========================================================
# Cell 7 — Geração filtrada (accept-reject) até N exato por classe
# ==========================================================
def generate_exact_filtered_for_class(class_name: str, n_final: int, gen_batch: int):
    print(f"\n=== Classe {class_name} | alvo final = {n_final} ===")
    out_dir = OUT_ROOT / class_name
    out_dir.mkdir(parents=True, exist_ok=True)

    existing = sorted(out_dir.glob("*.png"))
    kept = len(existing)
    attempts = 0

    print(f"  retomando: {kept}/{n_final} já estão salvas em {out_dir}")

    if kept >= n_final:
        print("  já completou, pulando.")
        return {
            "class_name": class_name,
            "n_final": n_final,
            "attempts": 0,
            "kept": kept,
            "accept_rate": None,
            "note": "já completo ao retomar"
        }

    # 1) Carregar DDPM EMA
    ema_network, gdf_util = load_ema_network_for_class(class_name)

    # 2) Carregar filtro 64
    index, k, thr_dist, meta = load_filter(class_name)
    print(f"Filtro 64: k={k}, thr_dist={thr_dist:.6f}, thr_quality={meta.get('threshold_quality', None)}")

    start = time.time()

    while kept < n_final:
        need = n_final - kept
        b = min(gen_batch, need)

        # gerar
        imgs = ddpm_sample_batch(ema_network, gdf_util, num_img=b)
        attempts += b

        # filtrar
        feats = extract_features_from_ddpm_uint8(imgs)          # [B,2048]
        mean_dist = knn_mean_distance(index, feats, k=k)        # [B]
        passes = mean_dist <= thr_dist

        # salvar aprovadas
        for i in range(b):
            if kept >= n_final:
                break
            if passes[i]:
                fname = f"{class_name}_{kept:06d}.png"
                save_uint8_image(imgs[i], out_dir / fname)
                kept += 1

        if kept % 200 == 0 or kept == n_final:
            elapsed = time.time() - start
            rate = kept / max(attempts, 1)
            print(f"  mantidas={kept}/{n_final} | tentadas={attempts} | taxa={rate:.3f} | {elapsed/60:.1f} min")

    report = {
        "class_name": class_name,
        "n_final": n_final,
        "attempts": attempts,
        "kept": kept,
        "accept_rate": float(kept / max(attempts, 1)),
        "k": k,
        "threshold_dist": thr_dist,
        "threshold_quality": float(1.0 / (1.0 + thr_dist)),
    }
    with open(out_dir / "generation_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"[DONE] {class_name} -> {kept} imagens em {out_dir}")
    return report


In [7]:
# ==========================================================
# Cell 8 — Rodar para todas as classes + relatório global
# ==========================================================
all_reports = {}

for cls in classes:
    rep = generate_exact_filtered_for_class(cls, N_FINAL_PER_CLASS, GEN_BATCH)
    all_reports[cls] = rep

with open(OUT_ROOT / "generation_report_all_classes.json", "w", encoding="utf-8") as f:
    json.dump(all_reports, f, ensure_ascii=False, indent=2)

print("\nConcluído. Dataset final em:", OUT_ROOT.resolve())


=== Classe BULKCARRIER | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-FILTERED-KNN-54K/BULKCARRIER
  já completou, pulando.

=== Classe CONTAINERSHIP | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-FILTERED-KNN-54K/CONTAINERSHIP
  já completou, pulando.

=== Classe GENERALCARGO | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-FILTERED-KNN-54K/GENERALCARGO
  já completou, pulando.

=== Classe OILPRODUCTSTANKER | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-FILTERED-KNN-54K/OILPRODUCTSTANKER
  já completou, pulando.

=== Classe PASSENGERSSHIP | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-FILTERED-KNN-54K/PASSENGERSSHIP
  já completou, pulando.

=== Classe TANKER | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-FILTERED-KNN-54K/TANKER
  já completou, pulando.

=== Classe TRAWLER | alvo final = 5400 ===
  retomando: 5400/5400 já estão salvas em DDPM-F

/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['image_input', 'time_input']. Received: the structure of inputs=('*', '*')
  warnings.warn(
I0000 00:00:1767616209.033842 1350036 service.cc:145] XLA service 0x7fb2dc03d740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767616209.033879 1350036 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-01-05 12:30:09.123763: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-05 12:30:09.445907: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8906
I0000 00:00:1767616209.794905 1350171 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_88', 8 byt

  mantidas=5200/5400 | tentadas=224 | taxa=23.214 | 5.5 min


I0000 00:00:1767616818.371710 1831782 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_4', 172 bytes spill stores, 172 bytes spill loads

I0000 00:00:1767616818.646215 1831773 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_92', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1767616818.669941 1831778 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_88', 116 bytes spill stores, 116 bytes spill loads

I0000 00:00:1767616875.895138 1868902 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_117', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1767616876.050924 1868888 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_117', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1767616876.390843 1868897 asm

  mantidas=5400/5400 | tentadas=427 | taxa=12.646 | 12.1 min
[DONE] YACHT -> 5400 imagens em DDPM-FILTERED-KNN-54K/YACHT

Concluído. Dataset final em: /tf/2025/DDPM-FILTERED-KNN-54K
